# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pangakeerthi11/keerthi-flyrank-ml-internship-2026/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import pandas as pd
from pathlib import Path

# Load the dataset already used in Section 2 with fallback
data_path = Path("data/raw/content_refresh_anonymized.csv")

if data_path.exists():
    signal_df = pd.read_csv(data_path)
else:
    url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    signal_df = pd.read_csv(url)

# Ensure the columns needed for this analysis are available
required_cols = [
    "avg_position",
    "ctr",
    "days_since_last_update"
]

missing_cols = [
    col for col in required_cols
    if col not in signal_df.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Convert analysis columns to numeric values
for col in required_cols:
    signal_df[col] = pd.to_numeric(
        signal_df[col], errors="coerce"
    )

# Signal 1: CTR in relation to search position
ctr_df = signal_df.dropna(
    subset=["avg_position", "ctr"]
).copy()

ctr_df = ctr_df[
    (ctr_df["avg_position"] > 0) &
    (ctr_df["ctr"] >= 0)
].copy()

ctr_df["position_bucket"] = pd.cut(
    ctr_df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "Above 20"]
)

ctr_table = (
    ctr_df.groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL 1: CTR by search-position bucket")
display(ctr_table)

# Signal 2: Content freshness / staleness
fresh_df = signal_df.dropna(
    subset=["days_since_last_update"]
).copy()

fresh_df = fresh_df[
    fresh_df["days_since_last_update"] >= 0
].copy()

fresh_df["freshness_bucket"] = pd.cut(
    fresh_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days",
            "91-180 days", "Above 180 days"]
)

fresh_table = (
    fresh_df.groupby("freshness_bucket", observed=False)
    .agg(
        n=("days_since_last_update", "size"),
        mean_days_old=("days_since_last_update", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2: Content freshness buckets")
display(fresh_table)

SIGNAL 1: CTR by search-position bucket


,position_bucket,n,mean_ctr
0,1-3,1141,2.714303
1,4-10,11842,0.651045
2,11-20,7273,0.323443
3,Above 20,8539,0.211333



SIGNAL 2: Content freshness buckets


,freshness_bucket,n,mean_days_old
0,0-30 days,20480,18.540918
1,31-90 days,175,53.628571
2,91-180 days,9171,104.106204
3,Above 180 days,174,224.643678


### Observed Signal Results and Verdicts

**Signal 1: CTR in relation to search position**

The observed bucket table shows that mean CTR decreases as search position moves from the top results to positions beyond 20. The mean CTR is approximately 2.7143 for positions 1–3, 0.6510 for positions 4–10, 0.3234 for positions 11–20, and 0.2113 for positions above 20. The sample counts are 1,141, 11,842, 7,273, and 8,539, respectively.

**Verdict: CONFIRMED.** The observed data supports an association between search position and CTR. However, this association alone does not prove that a title or snippet change will improve CTR.

**Signal 2: Content freshness and staleness**

The freshness table shows that the dataset contains content with different ages. The largest bucket contains 20,480 records updated within 30 days, while 9,171 records fall into the 91–180-day bucket. Only 175 records fall into the 31–90-day bucket, and 174 records are older than 180 days.

**Verdict: MIXED.** The table confirms that content ages vary, but age alone does not establish whether older content has worse search performance. Comparing performance across freshness buckets would provide stronger evidence before using staleness as a priority signal.

These verdicts are based on the observed bucket tables. The scoring rule is a transparent decision-support baseline, and its recommendations should be reviewed rather than treated as guaranteed improvements.

### My Rule and Its Reason Codes

The objective of this baseline is to identify content records that may need attention based on their search performance. Before creating the scoring rule, I will examine two signals using bucket tables and sample counts (`n`).

The first signal will examine click-through rate (CTR) in relation to search position. This helps identify records that may have an opportunity for a CTR improvement. The second signal will examine content freshness or staleness, which may help identify records that need a refresh review.

I will assign each signal one verdict: **CONFIRMED, OPPOSITE, MIXED, or FALSE**, based on the observed results. I will not assume that a signal is useful before checking the data.

The baseline will assign each eligible record a score, one reason code, and one action label. Higher scores will indicate higher priority for review. This is a simple decision-support rule, not a guarantee of improved search performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
print("Queue rows:", len(queue))
print("Queue columns:", queue.columns.tolist())
display(queue.head(5))
print("Reason codes:")
print(queue["reason_code"].value_counts())
print("Actions:")
print(queue["action"].value_counts())

Queue rows: 28795
Queue columns: ['rank', 'content_id', 'score', 'reason_code', 'action', 'confidence_note', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']


,rank,content_id,score,reason_code,action,confidence_note,impressions_90d,days_since_last_update,avg_position,ctr
0,1,content_fd16e3475c29,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,429,183,9.0,0.0
1,2,content_ea41fe5cf292,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,265,183,8.0,0.0
2,3,content_958a46db26bd,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,198,183,6.7,0.0
3,4,content_02b0d6e30129,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,176,313,6.9,0.0
4,5,content_f488400fca67,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,155,305,5.7,0.0


Reason codes:
reason_code
visible_content_review    18644
ctr_opportunity_review     7685
general_content_review     2308
stale_content_review        158
Name: count, dtype: int64
Actions:
action
Monitor performance         20952
Review title and snippet     7685
Review content freshness      158
Name: count, dtype: int64


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load the anonymized starter dataset
data_path = Path("data/raw/content_refresh_anonymized.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

# Check required columns
required = [
    "content_id",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the dataset schema before continuing."
    )

# Keep records with sufficient observable information
queue = df.copy()

for col in [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]:
    queue[col] = pd.to_numeric(queue[col], errors="coerce")

queue = queue.dropna(
    subset=[
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
    ]
)

queue = queue[
    (queue["impressions_90d"] > 0)
    & (queue["avg_position"] > 0)
    & (queue["ctr"] >= 0)
].copy()

# Create transparent baseline signals
queue["staleness_points"] = np.where(
    queue["days_since_last_update"] >= 180, 40,
    np.where(queue["days_since_last_update"] >= 90, 20, 0)
)

queue["visibility_points"] = np.where(
    queue["impressions_90d"] >= 500, 30,
    np.where(queue["impressions_90d"] >= 100, 15, 0)
)

queue["ctr_points"] = np.where(
    (queue["avg_position"] <= 10) & (queue["ctr"] < 0.05), 30,
    np.where(
        (queue["avg_position"] <= 20) & (queue["ctr"] < 0.03),
        20,
        0
    )
)

# Total score: higher means higher review priority
queue["score"] = (
    queue["staleness_points"]
    + queue["visibility_points"]
    + queue["ctr_points"]
)

# Assign exactly one main reason code
queue["reason_code"] = np.select(
    [
        queue["staleness_points"] >= 40,
        queue["ctr_points"] > 0,
        queue["visibility_points"] > 0,
    ],
    [
        "stale_content_review",
        "ctr_opportunity_review",
        "visible_content_review",
    ],
    default="general_content_review",
)

# Assign an action label
queue["action"] = np.select(
    [
        queue["reason_code"] == "stale_content_review",
        queue["reason_code"] == "ctr_opportunity_review",
    ],
    [
        "Review content freshness",
        "Review title and snippet",
    ],
    default="Monitor performance",
)

# Rank the queue
queue = queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Confidence notes reflect evidence volume, not proven accuracy
queue["confidence_note"] = np.where(
    queue["impressions_90d"] >= 500,
    "More search-impression evidence; recommendation still needs review",
    "Limited impression evidence; treat recommendation cautiously"
)

# Export only non-sensitive fields needed for the review
output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "confidence_note",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
queue[output_columns].to_csv(output_path, index=False)

print("Eligible records:", len(queue))
print("CSV saved to:", output_path)
display(queue[output_columns].head(10))

Dataset shape: (30000, 44)
Available columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Eligible records: 28795
CSV saved to: work/outputs/baseline_action_score.csv


,rank,content_id,score,reason_code,action,confidence_note,impressions_90d,days_since_last_update,avg_position,ctr
0,1,content_fd16e3475c29,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,429,183,9.0,0.00
1,2,content_ea41fe5cf292,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,265,183,8.0,0.00
2,3,content_958a46db26bd,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,198,183,6.7,0.00
3,4,content_02b0d6e30129,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,176,313,6.9,0.00
4,5,content_f488400fca67,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,155,305,5.7,0.00
5,6,content_ab27c30d81f4,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,103,304,8.9,0.00
6,7,content_c8e9d6ab9013,80,ctr_opportunity_review,Review title and snippet,More search-impression evidence; recommendatio...,208678,104,9.7,0.00
7,8,content_c1fe78bc4e37,80,ctr_opportunity_review,Review title and snippet,More search-impression evidence; recommendatio...,134055,104,7.5,0.03
8,9,content_4a6607efcb46,80,ctr_opportunity_review,Review title and snippet,More search-impression evidence; recommendatio...,128068,104,2.2,0.01
9,10,content_b115f7c74779,80,ctr_opportunity_review,Review title and snippet,More search-impression evidence; recommendatio...,123469,104,8.0,0.03


### Baseline Scoring Approach

I will implement one transparent scoring rule using the signals examined in Section 1. Each eligible record will receive a numerical score, one reason code explaining the main reason for its ranking, and one action label describing the recommended next step.

I will sort the records from highest to lowest score and export the ranked queue to `work/outputs/baseline_action_score.csv`.

The scoring rule will use only information available at the time of the decision. It will exclude future-window measurements, label-derived fields, and existing product flags as scoring inputs. The exported queue will serve as a baseline that a later machine-learning model can be compared against.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

queue_path = Path("work/outputs/baseline_action_score.csv")

if not queue_path.exists():
    raise FileNotFoundError(
        "Run Section 2 first to create the ranked queue."
    )

queue = pd.read_csv(queue_path)

top20 = queue.head(20).copy()

# Explain what could make each recommendation wrong
def explain_risk(row):
    risks = []

    if row["impressions_90d"] < 500:
        risks.append("Low impression volume may make the signal unreliable")

    if row["reason_code"] == "stale_content_review":
        risks.append("Old content may still be accurate and useful")

    if row["reason_code"] == "ctr_opportunity_review":
        risks.append("CTR may reflect search intent or snippet context")

    if row["reason_code"] == "visible_content_review":
        risks.append("Visibility alone does not prove a content problem")

    if not risks:
        risks.append("Additional context may change the recommended action")

    return "; ".join(risks)

top20["what_could_make_it_wrong"] = top20.apply(
    explain_risk, axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_could_make_it_wrong",
]

print("Top-20 baseline review")
display(top20[review_columns])

# Save a copy of the review
review_path = Path("work/outputs/top20_review.csv")
top20[review_columns].to_csv(review_path, index=False)

print("Top-20 review saved to:", review_path)


Top-20 baseline review


,rank,content_id,action,reason_code,confidence_note,what_could_make_it_wrong
0,1,content_fd16e3475c29,Review content freshness,stale_content_review,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
1,2,content_ea41fe5cf292,Review content freshness,stale_content_review,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
2,3,content_958a46db26bd,Review content freshness,stale_content_review,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
3,4,content_02b0d6e30129,Review content freshness,stale_content_review,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
4,5,content_f488400fca67,Review content freshness,stale_content_review,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
5,6,content_ab27c30d81f4,Review content freshness,stale_content_review,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
6,7,content_c8e9d6ab9013,Review title and snippet,ctr_opportunity_review,More search-impression evidence; recommendatio...,CTR may reflect search intent or snippet context
7,8,content_c1fe78bc4e37,Review title and snippet,ctr_opportunity_review,More search-impression evidence; recommendatio...,CTR may reflect search intent or snippet context
8,9,content_4a6607efcb46,Review title and snippet,ctr_opportunity_review,More search-impression evidence; recommendatio...,CTR may reflect search intent or snippet context
9,10,content_b115f7c74779,Review title and snippet,ctr_opportunity_review,More search-impression evidence; recommendatio...,CTR may reflect search intent or snippet context


Top-20 review saved to: work/outputs/top20_review.csv


### Top-20 Review

I will review the 20 highest-ranked records in the baseline queue. For each record, I will include the recommended action, its reason code, a confidence note, and an explanation of what could make the recommendation incorrect.

The review will help identify cases where the scoring rule produces weak or misleading recommendations. A high score does not necessarily mean that a record requires an intervention; additional context may be needed before taking action.

This review will help assess whether the baseline produces understandable and useful recommendations.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
from pathlib import Path
import pandas as pd

queue_path = Path("work/outputs/baseline_action_score.csv")
review_path = Path("work/outputs/top20_review.csv")

if not queue_path.exists() or not review_path.exists():
    raise FileNotFoundError(
        "Run Sections 2 and 3 before running Section 4."
    )

queue = pd.read_csv(queue_path)
top20 = pd.read_csv(review_path)

# Bring back 'impressions_90d' from the full queue to inspect weak picks
top20 = top20.merge(queue[["content_id", "impressions_90d", "score"]], on="content_id", how="left")

# Find potentially weak picks in the top 20
weak_picks = top20[
    (top20["impressions_90d"] < 500)
    | (top20["confidence_note"].str.startswith("Limited", na=False))
].copy()

print("Potentially weak picks in the Top 20:", len(weak_picks))

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "rank",
                "content_id",
                "score",
                "reason_code",
                "action",
                "confidence_note",
                "what_could_make_it_wrong",
            ]
        ]
    )
else:
    print("No low-volume picks were identified by this check.")

# Leakage audit: inspect columns actually used by the score
scoring_inputs = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

# These fields must not be scoring inputs
forbidden_patterns = [
    "label",
    "target",
    "future",
    "next_",
    "product_flag",
    "decision_flag",
]

suspect_inputs = [
    col for col in scoring_inputs
    if any(pattern in col.lower() for pattern in forbidden_patterns)
]

print("\nScoring inputs:", scoring_inputs)
print("Potentially unsafe scoring inputs:", suspect_inputs)

assert not suspect_inputs, (
    f"Review these potentially unsafe inputs: {suspect_inputs}"
)

print(
    "PASS: No label-, future-, or product-flag-named columns "
    "are used in the declared scoring inputs."
)

print(
    "LIMITATION: This name-based check is not proof of no leakage. "
    "Confirm the data windows and column definitions as well."
)

Potentially weak picks in the Top 20: 6


,rank,content_id,score,reason_code,action,confidence_note,what_could_make_it_wrong
0,1,content_fd16e3475c29,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
1,2,content_ea41fe5cf292,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
2,3,content_958a46db26bd,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
3,4,content_02b0d6e30129,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
4,5,content_f488400fca67,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...
5,6,content_ab27c30d81f4,85,stale_content_review,Review content freshness,Limited impression evidence; treat recommendat...,Low impression volume may make the signal unre...



Scoring inputs: ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr']
Potentially unsafe scoring inputs: []
PASS: No label-, future-, or product-flag-named columns are used in the declared scoring inputs.
LIMITATION: This name-based check is not proof of no leakage. Confirm the data windows and column definitions as well.


### Weak Picks and Leakage Check

I will inspect weak or questionable recommendations and explain why each one may be incorrect. Possible reasons include incomplete information, unusual search behaviour, or a signal that does not reliably indicate an opportunity for improvement.

I will also check that the baseline does not use future-window measurements, label-derived fields, or existing product flags as inputs to the scoring rule.

The final baseline will be treated as a transparent, directional prioritisation method. Its results will need to be evaluated on the same held-out data before comparing it fairly with a future machine-learning model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.